In [81]:
import os
import random

import pandas as pd
from scipy import stats

from tqdm.notebook import tqdm

from ir_rpp.pref_eval.pref_eval import get_measures, get_prefs
from ir_rpp.pref_eval.util import trec_io

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
DATA_DIR = "../../data/libraryThing"
SCORES = ['rpp', 'dcgrpp', 'invrpp', 'ap', 'ndcg', 'rr']


qrels = f"{DATA_DIR}/qrels/2018.txt"
all_runfiles = [ f"{DATA_DIR}/runs/{fname}" for fname in os.listdir(f"{DATA_DIR}/runs")]
assert len(all_runfiles) >= 2, "need at least two runs for comparison"
        
measures = []
measure_set = "all" # TODO: and `measures` to choose only one ?
measures = get_measures(measures,measure_set)

query_eval_wanted = True

binary_relevance = 4
relevance_floor = None 
topk = None

query_fraction = 1.0
label_fraction = 1.0
label_fraction_policy = "random"

SAMPLE = 0
qrels = trec_io.read_qrels(qrels,binary_relevance,relevance_floor)

In [ ]:
nb_comparisons = int(len(all_runfiles) * (len(all_runfiles) - 1) / 2)
pbar = tqdm(total=nb_comparisons)

scores_df_pairs = []
for i, runfile1 in enumerate(all_runfiles):
    for runfile2 in all_runfiles[i+1:]:
        runfiles = [runfile1, runfile2]
        
        # Run main of pref_eval
        if query_fraction < 1.0:
            qids = list(qrels.keys())
            num_sample = max(int(len(qids) * query_fraction),1)
            if num_sample < len(qids):
                qids_to_remove = random.sample(qids,len(qids)-num_sample)
                for qid in qids_to_remove:
                    qrels.pop(qid,None)
        if label_fraction < 1.0:
            if (label_fraction_policy == "pool"):
                qrels_pool_frequencies = trec_io.compute_qrel_pool_frequencies(runfiles,qrels)
            for qid in qrels.keys():
                dids = list(qrels[qid].keys())
                num_sample = max(int(len(dids) * label_fraction),1)
                if num_sample < len(dids):
                    if (label_fraction_policy == "pool"):
                        dids_to_remove = qrels_pool_frequencies[qid][num_sample:]
                    else:
                        dids_to_remove = random.sample(dids,len(dids)-num_sample)
                    for did in dids_to_remove:
                        qrels[qid].pop(did,None)

        runs:dict[str,trec_io.Run] = {}

        for runfile in runfiles:
            runid,run = trec_io.read_run(runfile,qrels,topk)
            runs[runid] = run

        all_prefs = get_prefs(SAMPLE,runs,measures,query_eval_wanted)

        # Do stats
        scores = []
        for line in all_prefs:
            if "run" in line or line.get("qid") == "all":
                continue
            scores.append(line)
        scores_df = pd.DataFrame(scores)
        scores_df_pairs.append(scores_df)
        
        pbar.update()


  0%|          | 0/210 [00:00<?, ?it/s]

In [ ]:
import pickle as pkl

with open("./results/scores_df_pairs.pkl", "wb") as file:
    pkl.dump(scores_df_pairs,file)

In [ ]:
count_significant = {score : 0 for score in SCORES}
PVAL_THRESHOLD = 0.05 / nb_comparisons # Bonferroni

for scores_df in tqdm(scores_df_pairs):
    for score in SCORES:
        res = stats.ttest_1samp(scores_df[score], popmean=0)
        if res.pvalue < PVAL_THRESHOLD:
            count_significant[score] += 1

  0%|          | 0/210 [00:00<?, ?it/s]

In [79]:
prop_significant = {score: round(count / nb_comparisons,4) for score, count in count_significant.items()}
print(prop_significant)

{'rpp': 0.981, 'dcgrpp': 0.9857, 'invrpp': 0.9857, 'ap': 0.9619, 'ndcg': 0.9857, 'rr': 0.9381}


In [87]:
print(len(scores_df["qid"].unique()))

7227
